In [ ]:
import os, sys
import pandas as pd
import numpy as np

from utils import (
    get_p2i
)

In [ ]:
def get_white_subjects(ethnicity_df):
    ethnicity_coding = pd.read_csv("/homes/bonazzola/ukbb_helpers/codings/coding1001.tsv", sep='\t')
    ethnicity_df.columns = range(1, 5)
    white_coding = ethnicity_coding.query("coding.astype('str').str.startswith('1')").coding
    return ethnicity_df.index[ethnicity_df.isin(set(white_coding)).any(axis=1)].to_list()

ETHNICITY_FILE = "/nfs/research/birney/controlled_access/ukb-cnv/data_fetch/baskets/2017133/self_reported_ethnicity_21000.txt"
ethnicity_df = pd.read_csv(ETHNICITY_FILE, sep='\t').set_index("f.eid").astype("Int64")
white_subjects = get_white_subjects(ethnicity_df)
# white_subjects = set([str(x) for x in white_subjects]) 
white_subjects = set([int(x) for x in white_subjects]) 

In [ ]:
dataset, file_prefix = 'ukb_real_data', "ukb_real_"
# dataset, file_prefix = 'ukb_simulated_data', ''

data_dir = os.path.join('data', dataset)
train_data = np.memmap(os.path.join(data_dir, f'{file_prefix}train.bin'), dtype=np.uint32, mode='r').reshape(-1, 3)
val_data = np.memmap(os.path.join(data_dir, f'{file_prefix}val.bin'), dtype=np.uint32, mode='r').reshape(-1, 3)

train_p2i = get_p2i(train_data)
val_p2i = get_p2i(val_data)

In [ ]:
def filter_subjects(data, include=None, exclude=None):

    if include is not None:
        filtered_data = data[np.isin(data[:,0], np.array(list(include), dtype=data.dtype))]

    if exclude is not None:
        filtered_data = data[~np.isin(data[:,0], np.array(list(exclude), dtype=data.dtype))]
        
    return filtered_data

In [ ]:
train_data = filter_subjects(train_data, white_subjects)

In [ ]:
train_data

In [ ]:
train_data.shape